# 06 — Submission formatting & validation

**Variant:** `robust_scale_log1p` · Align test predictions to `sample_submission.csv` ids and validate envelope.


In [ ]:
"""Shared imports for Rogii TVT pipeline notebooks."""
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NB_DIR = Path.cwd()
if not (NB_DIR / "phase_runner.py").is_file():
    NB_DIR = Path(r"/lustre/work/sweeden/agent-tracing-trace-baseline/examples/rogii/traces/preprocessing/robust_scale_log1p/notebooks")
VARIANT_DIR = NB_DIR.parent
ROGII_ROOT = Path("/lustre/work/sweeden/rogii")
sys.path.insert(0, str(NB_DIR))
if str(ROGII_ROOT) not in sys.path:
    sys.path.insert(0, str(ROGII_ROOT))

from phase_runner import (
    ARTIFACTS_ROOT,
    TRACE_INDEX,
    require_prior_phase,
    load_phase_manifest,
    trace_steps_for_phase,
    _resolve_path,
    _read_json,
    _load_train_predict,
)

pd.set_option("display.max_columns", 40)
plt.style.use("seaborn-v0_8-whitegrid")

PHASE = "06_submission"


## 1. Submission contract


In [ ]:
p01 = load_phase_manifest("01_data_analysis")
paths = _read_json(_resolve_path(p01["outputs"]["data_paths"]))
schema = _read_json(_resolve_path(p01["outputs"]["schema"]))
sample_sub = pd.read_csv(paths["sample_submission_csv"])
print("required columns:", schema["id_column"], schema["target_columns"])
print("submission rows expected:", len(sample_sub))
display(sample_sub.head())


## 2. Load model transform


In [ ]:
p04 = load_phase_manifest("04_model_training")
transform = _read_json(_resolve_path(p04["outputs"]["transform"]))
test_mat = np.load(_resolve_path(p04["outputs"]["test_preds_per_fold"]))
print("test fold preds shape:", test_mat.shape)


## 3. Mean test prediction vector


In [ ]:
test_mean = test_mat.mean(axis=1)
if transform.get("use_log1p"):
    test_pred = np.expm1(np.clip(test_mean, None, 20))
else:
    test_pred = test_mean
print("test_pred:", test_pred.shape, "min/max", np.nanmin(test_pred), np.nanmax(test_pred))


## 4. Build submission via phase_runner


In [ ]:
from phase_runner import run_06_submission
manifest = run_06_submission()
print(json.dumps(manifest, indent=2))


## 5. Validation report


In [ ]:
report = _read_json(_resolve_path(manifest["outputs"]["validation_report"]))
print(json.dumps(report, indent=2))
assert report.get("ok"), report


## 6. Submission preview


In [ ]:
sub = pd.read_csv(_resolve_path(manifest["outputs"]["submission_csv"]))
display(sub.head(10))
display(sub.tail(5))
print(sub.describe())


## 7. Target distribution vs train


In [ ]:
train_df = pd.read_csv(paths["train_csv"])
target = transform["target_column"]
train_df = _load_train_predict().align_train_target_to_schema(train_df, target)
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(train_df[target].dropna(), bins=40, alpha=0.5, label="train TVT", density=True)
ax.hist(sub[schema["target_columns"][0]], bins=40, alpha=0.5, label="submission", density=True)
ax.legend(); ax.set_title("Train vs submission TVT density")
plt.tight_layout(); plt.show()


## 8. Id alignment spot-check


In [ ]:
merged = sample_sub.merge(sub, on=schema["id_column"], suffixes=("_sample", "_pred"))
print("matched ids:", len(merged), "/", len(sample_sub))


## 9. Per-well prediction count


In [ ]:
well_prefix = sub[schema["id_column"]].astype(str).str.split("_").str[0]
print(well_prefix.value_counts().describe())


## 10. Submission file size and path


In [ ]:
sub_path = _resolve_path(manifest["outputs"]["submission_csv"])
print(sub_path, sub_path.stat().st_size, "bytes")


## 11. Negative / out-of-range submission check


In [ ]:
tcol = schema["target_columns"][0]
print("min:", sub[tcol].min(), "max:", sub[tcol].max(), "any_na:", sub[tcol].isna().any())


## 12. Row count contract


In [ ]:
assert len(sub) == len(sample_sub), (len(sub), len(sample_sub))
print("row count OK")


## 13. Kaggle submit command (manual)


```bash
kaggle competitions submit -c rogii-wellbore-geology-prediction \
  -f examples/rogii/traces/preprocessing/baseline_column_transformer/artifacts/06_submission/submission.csv \
  -m "baseline_column_transformer phase 06"
```


## 14. Pipeline complete


In [ ]:
state = _read_json(ARTIFACTS_ROOT / "pipeline_state.json")
print(json.dumps(state, indent=2))


## 15. Full artifact tree


In [ ]:
for phase_dir in sorted(ARTIFACTS_ROOT.iterdir()):
    if phase_dir.is_dir():
        files = list(phase_dir.glob("*"))
        print(phase_dir.name, "→", len(files), "files")


## 16. Trace pipeline summary


In [ ]:
total_steps = sum(len(trace_steps_for_phase(p)) for p in [
    "01_data_analysis", "02_statistical_framework", "03_feature_engineering",
    "04_model_training", "05_evaluation", "06_submission"])
print("total trace steps across 6 phases:", total_steps)


## 17. Reproducibility note

Full-scale training and submission should be executed via **Slurm** (`sbatch`) per project rules — this notebook validates the submission envelope on login node when artifacts exist.


## 18. Next steps

1. Submit via Kaggle CLI (section 13)
2. Compare duration track ablation grid from phase 02
3. Compare public LB score to phase 04/05 `cv_rmse`
